## Cell 1 Explanation
This cell verifies that the supervision package is installed and prints the active version.

In [1]:
import supervision as sv
print(sv.__version__)

0.27.0.post2


# Football Analysis Pipeline

This notebook processes football videos to detect players, track them, assign teams, and generate annotated output videos with bounding boxes, speed, and distance metrics.

## Setup
Make sure you have the required dependencies installed:
- ultralytics
- supervision
- opencv-python
- pandas
- numpy
- imageio

Place your input video in the `input_videos/` folder and your trained YOLO model in `models/best.pt`.

## How to Use
1. Place your video files in the `input_videos/` folder
2. Run the video selection cell below to choose your input video
3. Modify the configuration as needed
4. Run all cells in order
5. The annotated video will be saved to `output_videos/output_video.avi`

## Cell 3 Explanation
This cell imports all project modules and dependencies used by the analysis pipeline.

In [2]:
# Import required libraries
from utils import read_video, save_video
from trackers import Tracker
import cv2
import numpy as np
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator
import os
import glob

## Cell 4 Explanation
This cell lists available input videos and selects one safely using a bounded index.

In [3]:
# Video Selection
print("Available videos in input_videos/:")
video_files = glob.glob('input_videos/*.mp4') + glob.glob('input_videos/*.avi') + glob.glob('input_videos/*.mov')
for i, video in enumerate(video_files):
    print(f"{i+1}. {os.path.basename(video)}")

# Select video by index (change this number)
selected_index = 0  # Use 0 for first video, 1 for second, etc.

if video_files:
    selected_index = max(0, min(selected_index, len(video_files) - 1))
    INPUT_VIDEO_PATH = video_files[selected_index]
    print(f"\nSelected video: {os.path.basename(INPUT_VIDEO_PATH)}")
else:
    INPUT_VIDEO_PATH = 'input_videos/08fd33_4.mp4'  # Default
    print("No videos found, using default path")

Available videos in input_videos/:
1. 08fd33_4.mp4
2. E3c993bd2_0 (74).mp4

Selected video: 08fd33_4.mp4


## Cell 5 Explanation
This cell defines model, output, and caching configuration used by later processing steps.

In [4]:
# Configuration
MODEL_PATH = 'models/best.pt'
OUTPUT_VIDEO_PATH = 'output_videos/output_video.avi'
STUB_PATH = 'stubs/track_stubs.pkl'
READ_FROM_STUB = False  # Set to True to skip detection if stubs exist

print(f"Input: {INPUT_VIDEO_PATH}")
print(f"Output: {OUTPUT_VIDEO_PATH}")

Input: input_videos\08fd33_4.mp4
Output: output_videos/output_video.avi


## Cell 6 Explanation
This cell reads the selected input video and loads all frames into memory for processing.

In [5]:
# Step 1: Read the input video
print("Reading video...")
video_frames = read_video(INPUT_VIDEO_PATH)
print(f"✓ Loaded {len(video_frames)} frames")

Reading video...
Read 100 frames
Read 200 frames
Read 300 frames
Read 400 frames
Read 500 frames
Read 600 frames
Read 700 frames
Read 800 frames
Read 900 frames
EOF reached at frame 900
Total frames read: 900
✓ Loaded 900 frames


## Cell 7 Explanation
This cell initializes the tracker and runs object detection plus multi-object tracking.

In [6]:
# Step 2: Initialize tracker and get object tracks
print("Initializing tracker...")
tracker = Tracker(MODEL_PATH)

print("Getting object tracks...")
tracks = tracker.get_object_tracks(video_frames,
                                   read_from_stub=READ_FROM_STUB,
                                   stub_path=STUB_PATH)

Initializing tracker...
Getting object tracks...
[YOLO] Processing frames 1/900
[YOLO] Processing frames 2/900
[YOLO] Processing frames 3/900
[YOLO] Processing frames 4/900
[YOLO] Processing frames 5/900
[YOLO] Processing frames 6/900
[YOLO] Processing frames 7/900
[YOLO] Processing frames 8/900
[YOLO] Processing frames 9/900
[YOLO] Processing frames 10/900
[YOLO] Processing frames 11/900
[YOLO] Processing frames 12/900
[YOLO] Processing frames 13/900
[YOLO] Processing frames 14/900
[YOLO] Processing frames 15/900
[YOLO] Processing frames 16/900
[YOLO] Processing frames 17/900
[YOLO] Processing frames 18/900
[YOLO] Processing frames 19/900
[YOLO] Processing frames 20/900
[YOLO] Processing frames 21/900
[YOLO] Processing frames 22/900
[YOLO] Processing frames 23/900
[YOLO] Processing frames 24/900
[YOLO] Processing frames 25/900
[YOLO] Processing frames 26/900
[YOLO] Processing frames 27/900
[YOLO] Processing frames 28/900
[YOLO] Processing frames 29/900
[YOLO] Processing frames 30/900


## Cell 8 Explanation
This cell computes object positions from bounding boxes and pads missing track frames.

In [7]:
# Step 3: Add position tracking
tracker.add_position_to_tracks(tracks)

# Pad tracks to match video length
while len(tracks["players"]) < len(video_frames):
    tracks["players"].append({})
    tracks["ball"].append({})
    tracks["referees"].append({})

## Cell 9 Explanation
This cell sets up camera motion compensation and applies adjusted positions to tracked objects.

In [8]:
# Step 4: Camera movement estimation
print("Setting up camera movement...")
camera_movement_estimator = CameraMovementEstimator(video_frames[0])
camera_movement_per_frame = [[0, 0] for _ in range(len(video_frames))]
camera_movement_estimator.add_adjust_positions_to_tracks(tracks, camera_movement_per_frame)

Setting up camera movement...


## Cell 10 Explanation
This cell applies perspective transformation to map tracked positions into field coordinates.

In [9]:
# Step 5: View transformation
print("Applying view transformation...")
view_transformer = ViewTransformer()
view_transformer.add_transformed_position_to_tracks(tracks)

Applying view transformation...


## Cell 11 Explanation
This cell interpolates missing ball detections to create smoother ball trajectories across frames.

In [10]:
# Step 6: Interpolate ball positions
print("Interpolating ball positions...")
tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"])

Interpolating ball positions...


## Cell 12 Explanation
This cell estimates per-player speed and cumulative distance using transformed positions.

In [11]:
# Step 7: Speed and distance estimation
print("Estimating speed and distance...")
speed_and_distance_estimator = SpeedAndDistance_Estimator()
speed_and_distance_estimator.add_speed_and_distance_to_tracks(tracks)

Estimating speed and distance...


## Cell 13 Explanation
This cell learns team colors and assigns each tracked player to a team.

In [12]:
# Step 8: Team assignment
print("Assigning player teams...")
team_assigner = TeamAssigner()

# Find the first frame that actually has player detections
first_player_frame = None
for i, player_track in enumerate(tracks["players"]):
    if len(player_track) > 0:
        first_player_frame = i
        break

if first_player_frame is None:
    print("No player detections found in any frame – skipping team assignment.")
else:
    # Use that frame to learn team colors
    team_assigner.assign_team_color(
        video_frames[first_player_frame],
        tracks["players"][first_player_frame]
    )

    # Assign team + color for every frame
    for frame_num, player_track in enumerate(tracks['players']):
        for player_id, track in player_track.items():
            team = team_assigner.get_player_team(
                video_frames[frame_num],
                track['bbox'],
                player_id
            )
            tracks['players'][frame_num][player_id]['team'] = team
            tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors[team]


Assigning player teams...


## Cell 14 Explanation
This cell assigns ball possession to nearby players and records team-level control by frame.

In [13]:
# Step 9: Ball possession assignment
print("Assigning ball possession...")
player_ball_assigner = PlayerBallAssigner()
team_ball_control = []

for frame_num, player_track in enumerate(tracks['players']):
    ball_track = tracks['ball'][frame_num] if frame_num < len(tracks['ball']) else {}
    ball_bbox = ball_track.get(1, {}).get('bbox')
    assigned_player = player_ball_assigner.assign_ball_to_player(player_track, ball_bbox)

    if assigned_player != -1 and assigned_player in tracks['players'][frame_num]:
        tracks['players'][frame_num][assigned_player]['has_ball'] = True
        team_ball_control.append(tracks['players'][frame_num][assigned_player].get('team', 0))
    else:
        team_ball_control.append(0)

Assigning ball possession...


## Cell 15 Explanation
This cell normalizes team ball-control array length to match the number of video frames.

In [14]:
team_ball_control = np.array(team_ball_control).astype(int).reshape(-1)

if len(team_ball_control) < len(video_frames):
    padding = len(video_frames) - len(team_ball_control)
    team_ball_control = np.concatenate([team_ball_control, np.zeros(padding, dtype=int)])
elif len(team_ball_control) > len(video_frames):
    team_ball_control = team_ball_control[:len(video_frames)]

## Cell 16 Explanation
This cell draws player, ball, camera-motion, and speed/distance annotations on each frame.

In [15]:
# Step 10: Draw annotations on video frames
print("Drawing annotations...")
output_video_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control)

print("Drawing camera movement...")
output_video_frames = camera_movement_estimator.draw_camera_movement(output_video_frames, camera_movement_per_frame)

print("Drawing speed and distance...")
output_video_frames = speed_and_distance_estimator.draw_speed_and_distance(output_video_frames, tracks)

Drawing annotations...
Drawing camera movement...
Drawing speed and distance...


## Cell 17 Explanation
This cell writes the final annotated frames to the output video file.

In [16]:
# Step 11: Save the output video
print("Saving video...")
save_video(output_video_frames, OUTPUT_VIDEO_PATH)
print(f"✓ Video saved to {OUTPUT_VIDEO_PATH}")

Saving video...
FFmpeg not available, using OpenCV...
  Written 100/900 frames...
  Written 200/900 frames...
  Written 300/900 frames...
  Written 400/900 frames...
  Written 500/900 frames...
  Written 600/900 frames...
  Written 700/900 frames...
  Written 800/900 frames...
  Written 900/900 frames...
✓ Video saved to output_videos/output_video.avi
✓ Video saved to output_videos/output_video.avi


## Usage Instructions

**For Different Videos:**
1. Place your new video in the `input_videos/` folder
2. Change `selected_index` in the video selection cell (e.g., 0 for first video, 1 for second, etc.)
3. Re-run the notebook from the video selection cell onward
4. Optionally set `READ_FROM_STUB = True` for faster processing if the video content is similar

**Performance Tips:**
- Processing takes 10-15 minutes on CPU for a 30-second video
- Use `READ_FROM_STUB = True` after first run to skip detection
- For GPU acceleration, ensure CUDA is available